# 🇦🇺🇺🇸 US & ASX AI Quantitative Research Platform — Google Colab 主工作台

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davidshizw/stock-colab/blob/main/notebooks/01_Colab_Quickstart_End_to_End_Pipeline.ipynb)

本 Notebook 专为 **Google Colab** 环境设计，覆盖从 **美股（NYSE/Nasdaq）与澳股（ASX 200）数据入库、DuckDB + Parquet 持久化（挂载 Google Drive）、6 大类量化因子计算、跨市场隔夜联动（US Overnight → ASX）、含澳洲 Franking Credit 与美股预提税的组合优化与回测、AI 多智能体报告** 到 **Colab 内嵌交互式看板** 的完整闭环。

In [1]:
# @title 1. 环境初始化（解压最新内嵌源码 + 安装依赖）
import base64, io, os, subprocess, sys, tarfile
from pathlib import Path

payload = "..."
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(payload)), mode="r:gz") as tf:
    tf.extractall(".")
print("✅ Synced latest quant_platform source package into Colab workspace")

if "google.colab" in sys.modules:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "duckdb>=1.0.0", "pyarrow>=16.0.0", "pydantic>=2.7.0", "pydantic-settings>=2.2.0",
        "scipy>=1.13.0", "scikit-learn>=1.5.0", "lightgbm>=4.3.0", "yfinance>=0.2.40",
        "fastapi>=0.111.0", "uvicorn>=0.30.0", "httpx>=0.27.0"
    ])
if str(Path("src").resolve()) not in sys.path:
    sys.path.insert(0, str(Path("src").resolve()))

from quant_platform.colab.runtime import ColabSessionManager
colab_mgr = ColabSessionManager()
print("Colab Runtime Status:", colab_mgr.get_compute_unit_status())


Hardware Profile: {'in_colab': False, 'device': 'cpu', 'accelerator_name': 'CPU', 'tier': 'CPU', 'vram_gb': 0.0, 'estimated_cu_per_hour': 0.08}
Compute Unit & Storage Status: {'accelerator': 'CPU', 'tier': 'CPU', 'in_colab': False, 'storage_root': 'data_store', 'cu_rate_per_hour': 0.08, 'session_elapsed_minutes': 0.0, 'session_cu_used': 0.0, 'monthly_cu_budget': 1000.0, 'total_cu_used': 0.0, 'remaining_cu': 1000.0}


In [2]:
# 2. 初始化 DuckDB + Parquet 存储并执行美股与澳股数据入库
# 支持 mode='fixture' (高保真确定性跨市场测试数据) 或 mode='yfinance' (实时拉取 Yahoo Finance 数据)
from datetime import date, timedelta
from quant_platform.data.store import MarketDataStore
from quant_platform.data.ingestion import build_default_pipeline

store = MarketDataStore(db_path=colab_mgr.duckdb_path, parquet_dir=colab_mgr.parquet_dir)
DATA_MODE = 'yfinance'  # 可改为 'yfinance' 在 Colab 中拉取最新实时行情

end_date = date(2026, 11, 6)
start_date = end_date - timedelta(days=500)

pipeline = build_default_pipeline(store, mode=DATA_MODE)
ing_res = pipeline.run_ingestion(start_date=start_date, end_date=end_date, incremental=False)
print('Ingestion Summary:', ing_res)
latest_prices = store.conn.execute("SELECT symbol, market, trade_date, close, volume, source FROM daily_prices WHERE (symbol, trade_date) IN (SELECT symbol, MAX(trade_date) FROM daily_prices GROUP BY symbol) ORDER BY market, symbol").df()
display(latest_prices)


Ingestion Summary: {'run_id': 'ing-3daf2b76', 'instruments_upserted': 25, 'prices_upserted': 8879, 'fundamentals_upserted': 220, 'corporate_actions_upserted': 75, 'quality_report': {'checked_at_utc': '2026-11-08T20:31:36.701485Z', 'total_instruments': 25, 'total_price_rows': 8879, 'total_fundamental_rows': 220, 'missing_date_gaps': 0, 'stale_symbols': [], 'ohlc_violations': 0, 'provenance_sources': {'fixture': 8879}, 'status': 'HEALTHY'}}


In [3]:
# 3. 量化因子计算与美股 (US) / 澳股 (ASX) 系统化选股排名
import pandas as pd
from quant_platform.factors.engine import FactorEngine
from quant_platform.screener.ranker import EquityScreener

factor_engine = FactorEngine(store)
screener = EquityScreener(store, factor_engine)

us_top = screener.screen(as_of_date=end_date, market='US', top_n=8)
asx_top = screener.screen(as_of_date=end_date, market='ASX', top_n=8)

print('=== 美股 (US) 多因子排名 Top 8 ===')
display(pd.DataFrame(us_top)[['rank','symbol','name','sector','close','pe_ratio','roe','mom_6m','composite_score']])

print('\n=== 澳股 (ASX) 多因子排名 Top 8 (含澳洲 Franking Credit 税前等效股息率) ===')
display(pd.DataFrame(asx_top)[['rank','symbol','name','sector','close','pe_ratio','dividend_yield','franking_rate','franking_grossed_yield','composite_score']])

=== 美股 (US) 多因子排名 Top 8 ===


,rank,symbol,name,sector,close,pe_ratio,roe,mom_6m,composite_score
0,1,NVDA,NVIDIA Corp.,Information Technology,146.3769,41.60,0.6368,0.2164,0.2607
1,2,AAPL,Apple Inc.,Information Technology,219.6197,27.31,0.4865,0.1220,0.1194
2,3,JPM,JPMorgan Chase & Co.,Financials,218.9411,12.68,0.1827,0.0607,0.0750
3,4,CVX,Chevron Corp.,Energy,208.2286,13.91,0.1609,0.0857,0.0435
4,5,AMZN,Amazon.com Inc.,Consumer Discretionary,190.9079,35.67,0.2294,-0.0642,-0.0000
5,6,GOOGL,Alphabet Inc.,Communication Services,210.7508,21.86,0.3140,0.2286,-0.0000
6,7,LLY,Eli Lilly and Co.,Health Care,708.6080,52.12,0.4753,0.0342,-0.0000
7,8,XOM,Exxon Mobil Corp.,Energy,116.3134,14.06,0.1979,0.0039,-0.0435



=== 澳股 (ASX) 多因子排名 Top 8 (含澳洲 Franking Credit 税前等效股息率) ===


,rank,symbol,name,sector,close,pe_ratio,dividend_yield,franking_rate,franking_grossed_yield,composite_score
0,1,FMG.AX,Fortescue Ltd,Materials,30.0412,7.37,0.072,1.0,0.1029,0.7637
1,2,NAB.AX,National Australia Bank Ltd,Financials,35.1135,14.29,0.048,1.0,0.0686,0.1724
2,3,BHP.AX,BHP Group Ltd,Materials,48.5602,10.23,0.052,1.0,0.0743,0.1615
3,4,WBC.AX,Westpac Banking Corp,Financials,31.0746,12.86,0.051,1.0,0.0729,0.1414
4,5,CSL.AX,CSL Ltd,Health Care,331.7424,28.04,0.013,0.1,0.0136,0.0000
5,6,WDS.AX,Woodside Energy Group Ltd,Energy,38.6792,10.74,0.065,1.0,0.0929,0.0000
6,7,WES.AX,Wesfarmers Ltd,Consumer Discretionary,75.2128,24.89,0.031,1.0,0.0443,0.0000
7,8,CBA.AX,Commonwealth Bank of Australia,Financials,140.9229,17.61,0.038,1.0,0.0543,-0.0510


In [4]:
# 4. 跨市场联动分析 (US Overnight -> ASX Session) 与因子 IC / 分层检验
from quant_platform.cross_market.us_asx_analyzer import CrossMarketAnalyzer
from quant_platform.factors.evaluation import FactorEvaluator

cm = CrossMarketAnalyzer(store).analyze_overnight_transmission(start_date=start_date, end_date=end_date)
print(f"Zero Look-Ahead Alignment Verified: {cm['alignment_verified']} (Violations: {cm['lookahead_violations']})")
display(pd.DataFrame(cm['sector_sensitivity']))

ev = FactorEvaluator(store, factor_engine).evaluate_factors(start_date=start_date, end_date=end_date, market='ALL')
print('\n=== 因子 IC / Rank IC / 衰减与换手评估 ===')
display(pd.DataFrame(ev['factor_summary']))

Zero Look-Ahead Alignment Verified: True (Violations: 0)


,asx_sector,observations,corr_gap_SPY,corr_total_SPY,beta_SPY,corr_gap_QQQ,corr_total_QQQ,beta_QQQ,corr_gap_JPM,corr_total_JPM,beta_JPM,corr_gap_XOM,corr_total_XOM,beta_XOM,corr_gap_HG=F,corr_total_HG=F,beta_HG=F,corr_gap_AUDUSD=X,corr_total_AUDUSD=X,beta_AUDUSD=X
0,Consumer Discretionary,353,0.835,0.440,0.370,0.952,0.568,0.342,0.556,0.308,0.195,0.488,0.339,0.202,-0.160,-0.089,-0.052,0.025,0.041,0.044
1,Energy,353,0.465,0.261,0.362,0.380,0.229,0.227,0.332,0.165,0.172,0.440,0.262,0.258,0.890,0.614,0.598,0.704,0.465,0.820
2,Financials,353,0.856,0.750,0.577,0.692,0.594,0.326,0.682,0.612,0.353,0.534,0.497,0.270,0.104,0.103,0.056,0.445,0.364,0.355
3,Health Care,353,0.835,0.552,0.531,0.952,0.618,0.425,0.556,0.358,0.258,0.488,0.307,0.209,-0.160,-0.047,-0.031,0.025,0.151,0.184
4,Materials,353,0.448,0.419,0.582,0.366,0.338,0.336,0.315,0.271,0.283,0.434,0.388,0.382,0.891,0.778,0.759,0.780,0.683,1.208



=== 因子 IC / Rank IC / 衰减与换手评估 ===


,factor,mean_ic,mean_rank_ic,std_rank_ic,ic_ir,positive_ic_rate,bull_regime_rank_ic,bear_regime_rank_ic,avg_rank_turnover,decay_5d_ric,decay_10d_ric,decay_21d_ric
0,composite_score,-0.0411,-0.0322,0.2401,-0.134,0.533,-0.0438,-0.0190,0.093,0.0078,-0.0402,-0.0322
1,low_risk_score,-0.0352,-0.0608,0.2444,-0.249,0.333,-0.1056,-0.0096,0.077,0.0104,-0.0471,-0.0608
2,market_specific_score,-0.0401,-0.0508,0.1964,-0.259,0.333,-0.0163,-0.0901,0.300,0.0433,-0.0057,-0.0508
3,momentum_score,-0.1134,-0.1001,0.2224,-0.450,0.200,-0.0536,-0.1532,0.206,-0.0126,-0.0983,-0.1001
4,quality_score,0.0242,0.0026,0.1872,0.014,0.400,-0.0551,0.0685,0.039,-0.0298,-0.0740,0.0026
5,value_score,0.0157,-0.0027,0.2159,-0.013,0.600,-0.0196,0.0166,0.005,0.0519,0.0228,-0.0027


In [5]:
# 5. 策略回测、组合优化与 Colab 内嵌 ECharts 交互式看板渲染
from IPython.display import HTML, display
from quant_platform.backtest.engine import BacktestEngine
from quant_platform.schemas import BacktestConfig, CurrencyCode
from quant_platform.agents.coordinator import ResearchCoordinatorAgent

bt_cfg = BacktestConfig(
    strategy_name='US+ASX Multi-Factor Composite',
    market_filter='ALL',
    base_currency=CurrencyCode.AUD,
    start_date=start_date + timedelta(days=60),
    end_date=end_date,
    rebalance_frequency='M',
    top_n=6,
    weighting_scheme='risk_parity',
    include_franking_credits=True,
    apply_us_withholding_tax=True,
)
bt_res = BacktestEngine(store, factor_engine).run_backtest(bt_cfg)

# 在 Colab 单元格内直接渲染交互式 ECharts 投研看板
html_widget = colab_mgr.render_inline_colab_dashboard(
    us_top=us_top,
    asx_top=asx_top,
    backtest_result=bt_res.model_dump(mode='json'),
    cross_market=cm,
)
display(HTML(html_widget))

# 生成 AI 多智能体确定性溯源报告
report = ResearchCoordinatorAgent(store).generate_research_report(as_of_date=end_date, market_scope='ALL', focus_symbol='BHP.AX')
print('\n=== AI Research Coordinator Executive Summary ===')
print(report.executive_summary)

Rank,Symbol,Sector,Price,PE,ROE,Score
#1,NVDA,Information Technology,$146.38,41.6x,63.7%,+0.261
#2,AAPL,Information Technology,$219.62,27.3x,48.6%,+0.119
#3,JPM,Financials,$218.94,12.7x,18.3%,+0.075
#4,CVX,Energy,$208.23,13.9x,16.1%,+0.043
#5,AMZN,Consumer Discretionary,$190.91,35.7x,22.9%,-0.000
#6,GOOGL,Communication Services,$210.75,21.9x,31.4%,-0.000
Rank,Symbol,Sector,Price,CashDiv,GrossDiv,Score
#1,FMG.AX,Materials,A$30.04,7.20%,10.29%,+0.764
#2,NAB.AX,Financials,A$35.11,4.80%,6.86%,+0.172
#3,BHP.AX,Materials,A$48.56,5.20%,7.43%,+0.162



=== AI Research Coordinator Executive Summary ===
基于截至 2026-11-06 的严格无前视偏差（Point-in-Time）数据，多因子选股模型在 ALL 市场筛选出排名靠前的标的为：WDS.AX, FMG.AX, NVDA, WES.AX, JPM。核心关注标的 BHP.AX 的综合因子得分为 0.21，ROE 为 31.9%，含澳洲红利抵免（Franking Credit）/税后股息率为 7.43%。风险平价（Risk Parity）组合在回测期内实现年化收益（CAGR）10.47%，夏普比率 0.70，最大回撤 5.75%。


In [6]:
# 7. 全市场潜力股并发扫描 (108只美股 S&P500/Nasdaq100 + 澳股 ASX200 全行业黑马挖掘)
import pandas as pd
from quant_platform.screener.discovery import MarketWidePotentialScanner

scanner = MarketWidePotentialScanner(max_workers=16)
scan_res = scanner.scan_market(period='1y', top_k_per_market=15)
print(f"已并发扫描全市场标的: {scan_res['total_scanned']} 只 (美股 {scan_res['us_scanned']} 只, 澳股 {scan_res['asx_scanned']} 只)")

cols = ['rank', 'symbol', 'name', 'sector', 'close', 'forward_pe', 'roe', 'revenue_growth', 'earnings_growth', 'mom_6m', 'analyst_target_upside', 'after_tax_or_franking_yield', 'expected_6m_total_return', 'final_potential_score', 'catalyst_tag']
print('\n=== 🔥 美股 (US) 全市场潜力股 Top 15 ===')
display(pd.DataFrame(scan_res['us_potential_leaders'])[cols])

print('\n=== 🚀 澳股 (ASX 200) 全市场潜力股 Top 15 ===')
display(pd.DataFrame(scan_res['asx_potential_leaders'])[cols])


In [ ]:
# 8. Colab 量化引擎 <-> Grok (xAI) 实时 X 舆情与突发新闻串联研判
# 如需启用 Grok Live X Search 真机大模型，可在 Colab 左侧 🔑 Secrets 中添加 XAI_API_KEY，或直接填入下方变量：
XAI_API_KEY = ''  # @param {type:"string"}

from quant_platform.agents.grok_bridge import GrokColabBridge

grok_bridge = GrokColabBridge(api_key=XAI_API_KEY or None, model='grok-3-mini', enable_live_x_search=True)
top_candidates = scan_res['us_potential_leaders'][:6] + scan_res['asx_potential_leaders'][:6]
grok_Invalid_or_Valid = grok_bridge.enrich_candidates_with_grok(top_candidates, quant_weight=0.70, grok_weight=0.30)

print('🔗 当前串联引擎模式:', grok_Invalid_or_Valid['engine_mode'])
grok_cols = [
    'fused_rank', 'symbol', 'name', 'market', 'close',
    'final_potential_score', 'grok_sentiment_score', 'x_buzz_level',
    'grok_fused_score', 'grok_adjusted_6m_return', 'latest_headline', 'grok_risk_warning'
]
display(grok_Invalid_or_Valid['enriched_df'][grok_cols])
